# 独立实验解答

**注意**：本文件为单独提取出的练习代码（包含实验 A、B、C）。

如果在 Jupyter 环境中运行此文件，请确保已经执行了相关的前置依赖（如初始化 `spark` session、定义 `schema` 以及加载 `df_bronze` 数据）。

In [ ]:
# 如果运行报错 No module named 'pyspark'，请取消下面的注释并运行本单元格安装依赖
# %pip install pyspark

## 实验 A：数据质量审计 (统计每一列的 Null 数量)

In [ ]:
# 练习 A1: 统计 df_bronze 中每一列的 null 数量
import pyspark.sql.functions as F

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns
])
print("Bronze 表中各列的 null 数量统计：")
null_counts.show()


## 实验 B：增加新的清洗规则 (处理负数观看时长)

In [ ]:
# 练习 B1: 创建 raw_logs_v2.csv，包含原始数据，并额外加入负数观看时长行 (-20.0)
data_content_v2 = """event_id,user_id,video_id,duration_watched,device_type,event_timestamp
101,500,vid_a,120.5,iPhone,2023-01-01T10:00:00
102,501,vid_b,30.0,Android,2023-01-01T10:05:00
103,CORRUPT,vid_c,zero,iPhone,2023-01-01T10:10:00
104,502,vid_a,,iPad,2023-01-01T10:15:00
105,503,vid_d,300.0,Android,2023-01-01T10:20:00
106,504,vid_e,15.5,iPhone,2023-01-01T10:25:00
107,505,vid_f,-20.0,Android,2023-01-01T10:30:00
"""

with open("raw_logs_v2.csv", "w", encoding="utf-8") as f:
    f.write(data_content_v2)
print("包含负数观看时长的 raw_logs_v2.csv 已创建。\n")

# 练习 B2: 读取 raw_logs_v2.csv
df_bronze_v2 = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv("raw_logs_v2.csv")

# 练习 B3: 构建新的 Silver 表
df_silver_v2 = (
    df_bronze_v2
    .withColumn("timestamp", F.to_timestamp("event_timestamp"))
    .fillna(0, subset=["duration_watched"])
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("duration_watched") >= 0)
    .select(
        "event_id",
        "user_id",
        "video_id",
        "duration_watched",
        "device_type",
        "timestamp"
    )
)
print("新的 Silver 表 (移除了负数观看时长)：")
df_silver_v2.show(truncate=False)


## 实验 C：构建新的 Gold 表 (按视频统计)

In [ ]:
# 练习 C: 构建按 video_id 聚合的 Gold 表
df_gold_video_stats = (
    df_silver_v2
    .groupBy("video_id")
    .agg(
        F.count("*").alias("num_events"),
        F.sum("duration_watched").alias("total_duration"),
        F.avg("duration_watched").alias("avg_duration")
    )
    .orderBy(F.col("total_duration").desc())
)

print("新的 Gold 表 (按 video_id 聚合)：")
df_gold_video_stats.show()

print("Gold 表的物理执行计划：")
df_gold_video_stats.explain()
